# 详细文档: 通用框架 - 仿真控制器

**相关模块:** `common/controller.py`

## 目的
此文档旨在详细介绍`SimulationController`——整个模型框架的“大脑”。控制器负责管理一个由多个模型组件（如水文流域、水力河道、汇流点等）组成的网络，并按正确的顺序执行模拟。

此笔记本将：
1.  解释`SimulationController`的核心职责。
2.  演示如何使用`add_component`和`connect`方法来构建一个有向无环图（DAG）网络。
3.  解释控制器如何通过拓扑排序来确定DAG网络中组件的正确执行顺序。
4.  解释控制器如何检测网络中的环路，并使用迭代法来求解带环路的复杂网络。

## 1. 环境设置

我们导入控制器以及一些将要用作网络组件的模型。我们还包括了之前定义的`RoutingWrapper`，因为`MuskingumCungeRouting`本身不符合`BaseModelComponent`接口。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from common.base_model import BaseModelComponent
from common.controller import SimulationController
from common.junction import Junction
from preissmann_model.model import HydraulicModel
from hydro_model.routing import MuskingumCungeRouting

# --- 包装类定义 ---
class RoutingWrapper(BaseModelComponent):
    def __init__(self, name: str, routing_module):
        super().__init__(name)
        self.routing_module = routing_module
    def step(self, inflows: dict, dt: float):
        total_inflow = sum(inflows.values())
        self.outflow = self.routing_module.run(total_inflow)

# --- 辅助函数定义 ---
class Source(BaseModelComponent):
    def __init__(self, name: str, hydrograph: np.ndarray):
        super().__init__(name)
        self.hydrograph = hydrograph
        self.t = 0
    def step(self, inflows: dict, dt: float):
        self.outflow = self.hydrograph[self.t] if self.t < len(self.hydrograph) else 0
        self.t += 1

## 2. 案例一: 有向无环图 (DAG) 网络

最常见的河网是树状的，即有向无环图（Directed Acyclic Graph, DAG）。在这种网络中，水流方向是单一的，不存在循环。我们构建一个简单的Y型河网来说明。

### 2.1 构建网络
我们创建两个上游支流 `SourceA`, `SourceB`，一个汇流点 `J1`，和一个下游干流 `RouteC`。

In [ ]:
controller_dag = SimulationController()

# 创建组件
source_a = Source(name="SourceA", hydrograph=np.array([10, 20, 10]))
source_b = Source(name="SourceB", hydrograph=np.array([5, 10, 15]))
junction1 = Junction(name="J1")
route_c = RoutingWrapper(name="RouteC", routing_module=MuskingumCungeRouting(length=1000, slope=0.001, manning_n=0.03, width=20))

# 添加组件到控制器
controller_dag.add_component(source_a)
controller_dag.add_component(source_b)
controller_dag.add_component(junction1)
controller_dag.add_component(route_c)

# 连接组件
controller_dag.connect("SourceA", "J1")
controller_dag.connect("SourceB", "J1")
controller_dag.connect("J1", "RouteC")

print("Y型 (DAG) 河网构建完成。")

### 2.2 拓扑排序与执行顺序

在运行模拟之前，控制器需要确定一个正确的计算顺序，确保在计算一个组件之前，其所有上游组件都已经被计算过。这是通过**拓扑排序**实现的。我们可以调用内部方法 `_detect_and_sort_components` 来查看这个执行顺序。

In [ ]:
controller_dag._detect_and_sort_components()
print(f"\n检测到的执行顺序: {controller_dag.execution_order}")

可以看到，控制器正确地识别出了从上游到下游的计算顺序。`SourceA`和`SourceB`的顺序可能会变化，因为它们是并行的，但它们一定在`J1`之前，而`J1`一定在`RouteC`之前。

## 3. 案例二: 带环路的网络

更复杂的河网可能包含环路，例如分汊河道或水库调度形成的循环。对于这种情况，简单的拓扑排序不再适用。控制器能够检测到环路，并切换到迭代法进行求解。

### 3.1 构建环状网络
我们构建一个菱形的环状网络，其中水流在`J1_split`处分开，流经`B_branch`和`C_branch`，然后在`J2_merge`处汇合。

In [ ]:
controller_loop = SimulationController()

# 创建组件
source_loop = Source(name="A", hydrograph=np.array([0,0,0,10,25,50,70,60,45,30,20,15,10,5,2,1] + [0]*34))
j1_split = Junction(name="J1_split")
route_b = RoutingWrapper(name="B_branch", routing_module=MuskingumCungeRouting(length=4000, slope=0.0003, manning_n=0.03, width=20.0))
route_c = RoutingWrapper(name="C_branch", routing_module=MuskingumCungeRouting(length=4000, slope=0.0003, manning_n=0.05, width=25.0))
j2_merge = Junction(name="J2_merge")
route_d = RoutingWrapper(name="D_out", routing_module=MuskingumCungeRouting(length=3000, slope=0.0002, manning_n=0.03, width=40.0))

components = [source_loop, j1_split, route_b, route_c, j2_merge, route_d]
for comp in components:
    controller_loop.add_component(comp)

# 连接组件
controller_loop.connect("A", "J1_split")
controller_loop.connect("J1_split", "B_branch")
controller_loop.connect("J1_split", "C_branch")
controller_loop.connect("B_branch", "J2_merge")
controller_loop.connect("C_branch", "J2_merge")
controller_loop.connect("J2_merge", "D_out")

print("菱形环状河网构建完成。")

### 3.2 环路检测与迭代求解

我们再次调用 `_detect_and_sort_components`。这次，控制器应该能发现拓扑排序无法遍历所有节点，从而识别出哪些组件位于环路中。

In [ ]:
controller_loop._detect_and_sort_components()
print(f"\n检测到的环路内组件: {controller_loop.looped_components}")

控制器正确地识别出了所有参与环路计算的组件。

当调用`run`方法时，控制器会对这些`looped_components`进行特殊处理：在每个时间步内部，它会反复对这些组件进行迭代计算，直到所有组件的流量值都收敛（变化小于一个阈值），或者达到最大迭代次数。这样就确保了在环路中能得到一个物理上一致的解。

### 3.3 运行环路模拟
我们运行这个环状网络，并观察控制器的输出日志，可以看到它报告了迭代收敛的过程。

In [ ]:
num_steps = 50
history = []
history_generator = controller_loop.run(num_steps=num_steps, dt=1.0, global_inputs={})
for status in history_generator:
    pass # 在Jupyter中，print可以实时显示每个时间步的日志